# Running Processing Pipelines Tutorial 

_How to process L0 to L1B data, as well as make a flag map based on pre-existing data._

In [1]:
import numpy as np                # for scientific computing, good for working with large arrays  
import pandas as pd               # for data analysis, has easy to work with data frames 
import matplotlib.pyplot as plt   # for plotting 
from astropy.io import fits       # for reading / writing / manipulating FITS files 

import sys
sys.path.append('../') 

Most people will probably never run the processing pipelines themselves. But if you wanted to you could! 

The check_observation function may be useful even if you have no desire to reprocess data.

You will need to have the Github repo these tutorials reside in, m3-pipeline-dev, downloaded. 

In [2]:
# observation ID is the date, time, and mode designation unique to every observation in the M3 corpus 
# in every file name in the PDS this is the part that goes before _l0.fits, _l2_rfl.fits etc 
obs_id = "m3g20090420t191718" 

# local root, where data is stored 
local_root = './tutorial_data'

### Basic Observation Metadata for Processing 

If you want to know basic information like observation temperature or the flat field ID used in the original processing pipeline, the function check_observation can be helpful. 

Alternatively, you can load the observation metadata tables in /m3-pipeline-dev/l0_l1b_l2/cal_data with a tool like Pandas and sort for the observation IDs you are interested in. 

In [3]:
from m3_pipeline_tools.reference import check_observation   # import function for checking observation metadata 

# Check_observation returns a tuple where the first two entries are warnings & errors. Here we only care about the metadata dictionary.
_, _, metadata = check_observation(obs_id)  # pass obs ID string

Below we look at the dictionary object for this particular observation. 

In [4]:
metadata

{'obs_id': 'm3g20090420t191718',
 'obs_type': 'G',
 'date': '2009-04-20',
 'time': '19:17:18',
 'bde_id': 'm3g20090420t201515',
 'flat_id': 'm3g20090420t183130',
 'dark_id': 'm3g20090420t201515',
 'beta_angle': 55.44,
 'obs_temperature': 148.93,
 'lines': 844,
 'limb_dir': 'ascending',
 'yaw_dir': 'reverse',
 'dark_temp': 148.93,
 'bad_dark': False,
 'bad_flat': False,
 'bad_bde': False,
 'closest_dark': 'm3g20090420t201515',
 'dark_matches_closest': True,
 'closest_dark_lines': 232,
 'closest_dark_temp': 148.93,
 'closest_dark_gap_min': 57.95,
 'replacement_dark_id': 'm3g20090420t201515',
 'flat_temp': np.float64(148.93)}

When using an observation processed by the original mission, it's important to check that the bad_dark, bad_flat and bad_bde keys in the dictionary are not "True". This indicates a product that ought to be ignored due to an illuminated dark signal observation. 

### Data for Processing 

Currently the data you will need for processing (listed below) should all be stored in the same directory. 

_Note the below list is incomplete because we are not currently processing geometry data._

To process an observation to L1B using the "new" pipeline you will need: 
1) the L0 observation data
2) the L0 dark signal data ('dark_id') 
3) the BDE map ('bde_id') 

To process an observation to L1B using the "old" pipeline you will need: 
1) the L0 observation data
2) the L0 dark signal data ('dark_id') 
3) the BDE map ('bde_id') 
4) the observation-based flat field ('flat_id')

To make a flag backplane from an existing L1B product you will need: 
1) the L1B observation data
You should have but aren't required:
2) the L0 observation data (if this isn't present we can't flag pixels that are negative in L0)
3) the BDE map ('bde_id') (if this isn't present we can't flag what was spectrally interpolated)

_Technically we can make the pipeline work without the pre-existing BDE map but I haven't done that yet._

### L0 to L1B Pipeline 

In [5]:
from m3_pipeline_tools.pipeline import run_pipe  # import function to start pipeline 

You can run a 'mission' or 'new' version of the pipeline. I recommend the 'new' version. The mission version uses all of the mission-produced calibration files and tries to be as close to the DPSIS as possible (_But the DPSIS leaves room for interpretation_).

Currently 'new' also writes the new flat field generated by the pipeline to your local_root as a FITS file during processing. 

__Warning:__ Output L1B files are large, roughly double the size of L0 files. The max memory usage is also roughly 1.5-2x the L1B file size. So make sure you have room on your computer before running the pipeline below! 

#### Backplanes Included 
Produces observation data, a dark std image in detector POV, and the flag backplane. 

In [6]:
# run = 'new' 
    
# obs_image, dark, flag = run_pipe(
#     obs_id = obs_id.lower(), 
#     pipe_version= run,      # 'mission' or 'new' 
#     local_root= local_root, # where data required (listed above) is stored 
#     save_steps = False,     # True if we want the observation at each processing step to be written to a FITS file. Slows things down.  
#     backplanes = True,      # True if we want flag backplane and dark std backplane to be created 
#     smooth_shape = True,    # True if we want the mission's smooth shape correction applied during processing 
#     verbose = True,         # True tells us what the pipeline is doing in more detail while it runs 
# )

# """
# uncomment below if you wanted to write the outputs to FITS files
# """
# fits.writeto(f'{local_root}/{obs_id}_l1b_rdn_new', obs_image.transpose(1,0,2), overwrite=True)
# fits.writeto(f'{local_root}/{obs_id}_flags_new.fits', flag.transpose(1,0,2), overwrite=True)
# fits.writeto(f'{local_root}/{obs_id}_dark_std_new.fits', dark, overwrite=True)

# #these are big arrays so delete them between runs if doing multiple in a loop
# del obs_image
# del flag
# del dark

#### No Backplanes
If you run the pipeline with backplanes = False, the pipeline only outputs obs_image, not obs_image, dark, flag. 

In [7]:
# run = 'mission' 

# obs_image = run_pipe(
#     obs_id = obs_id.lower(), 
#     pipe_version= run,      # 'mission' or 'new' 
#     local_root= local_root, # where data required (listed above) is stored 
#     save_steps = False,     # True if we want the observation at each processing step to be written to a FITS file. Slows things down.  
#     backplanes = False,     # True if we want flag backplane and dark std backplane to be created 
#     smooth_shape = True,    # True if we want the mission's smooth shape correction applied during processing 
#     verbose = True,         # True tells us what the pipeline is doing in more detail while it runs 
# )

# # """
# # uncomment below if you wanted to write the outputs to FITS files
# # """
# fits.writeto(f'{local_root}/{obs_id}_l1b_rdn_new', obs_image.transpose(1,0,2), overwrite=True)

# #these are big arrays so delete them between runs if doing multiple in a loop
# del obs_image

### Flagging Pre-Existing L1B Data

We can flag an already existing L1B data set with the BDE map, L0 data, and L1B data. This outputs only a flag backplane. 

In [8]:
from m3_pipeline_tools.l1b_utils.make_backplanes import make_mission_l1b_flag_backplane  # import function for ONLY making flag map for L1B 

In [9]:
flag = make_mission_l1b_flag_backplane(
    obs_id = obs_id.lower(),
    local_root = local_root, 
    verbose=True,
)
    
fits.writeto(f'{local_root}/{obs_id}_flag.fits', flag, overwrite=True)
del flag